# Agent with Safety Guardrails

Deploying an agent in production requires more than a good system prompt. Users send unexpected inputs; LLMs occasionally hallucinate sensitive content. Guardrails enforce hard constraints at the Python layer — independently of what the LLM decides.

**What you'll build:** an agent wrapped with input and output validation that blocks harmful patterns, detects PII, and restricts off-topic queries — raising a clear error before any tool is called or any response is returned.

**Time:** ~15 min | **Difficulty:** Intermediate

**What you'll learn:**
- `ContentFilter` — block regex patterns and keyword lists in text
- `PIIDetector` — find emails, phone numbers, SSNs, and credit card numbers
- `TopicRestrictor` — allow or block topics by keyword
- `Guardrails` — compose multiple checks into one pipeline
- Pre-call input validation and post-call output validation patterns

## Prerequisites

You need:
- An **OpenAI API key** (`OPENAI_API_KEY`)
- The `synapsekit` package (installed below)

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key here
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Import guardrail classes

SynapseKit provides `ContentFilter`, `PIIDetector`, `TopicRestrictor`, and the `Guardrails` pipeline composer.

In [ ]:
import asyncio
from synapsekit.agents import (
    ContentFilter,
    FunctionCallingAgent,
    Guardrails,
    GuardrailResult,
    PIIDetector,
    TopicRestrictor,
    DuckDuckGoSearchTool,
    CalculatorTool,
)
from synapsekit.llms.openai import OpenAILLM

## Step 2: Configure individual checks

Each guardrail class has a standalone `.check(text)` method that returns a `GuardrailResult`. You can use them individually before composing them into a `Guardrails` pipeline.

In [ ]:
# Block content that matches security-sensitive patterns
content_filter = ContentFilter(
    blocked_patterns=[
        r"password\s*[:=]",    # credential leakage pattern
        r"\bAPI[_\s]?key\b",   # API key exposure
        r"<script[^>]*>",       # XSS attempt
    ],
    blocked_words=["hack", "exploit", "inject"],
    max_length=2000,           # reject excessively long inputs
)

# Detect personally identifiable information
pii_detector = PIIDetector(
    detect=["email", "phone", "ssn", "credit_card"],
)

# Keep the agent focused on its domain
topic_restrictor = TopicRestrictor(
    blocked_topics=["politics", "religion", "gambling"],
)

# Quick individual check
result = pii_detector.check("My email is alice@example.com")
print(f"PII check passed: {result.passed}")
print(f"Violations: {result.violations}")

## Step 3: Compose into a Guardrails pipeline

`Guardrails` runs all checks in order and collects every violation into a single `GuardrailResult`. This means one call gives you a complete picture of all problems rather than stopping at the first failure.

In [ ]:
input_guardrails = Guardrails(checks=[
    content_filter,
    pii_detector,
    topic_restrictor,
])

# Separate output guardrails — may differ from input rules
output_guardrails = Guardrails(checks=[
    PIIDetector(),                       # ensure LLM didn't echo PII back
    ContentFilter(blocked_words=["hack", "exploit"]),
])

print("Input guardrails pipeline configured with 3 checks.")
print("Output guardrails pipeline configured with 2 checks.")

## Step 4: Test guardrails independently

Always verify guardrail behavior before wiring them to an agent. A guardrail that blocks too aggressively is as harmful as one that blocks too little.

In [ ]:
test_cases = [
    ("What is 2 + 2?", False),                          # should pass
    ("My email is alice@example.com", True),             # PII violation
    ("password: secret123", True),                       # pattern violation
    ("Tell me about politics in the US", True),          # topic violation
    ("How do I hack a database?", True),                 # keyword violation
]

for text, expect_violation in test_cases:
    result = input_guardrails.check(text)
    status = "FAIL" if result.passed == expect_violation else "PASS"
    print(f"[{status}] {'BLOCKED' if not result.passed else 'ALLOWED'}: {text[:50]}")
    if not result.passed:
        for v in result.violations:
            print(f"        {v}")

## Step 5: Build a guardrailed agent wrapper

Wrap the agent's `run()` method to enforce guardrails without modifying the agent class itself. This keeps the validation logic separate and easily testable.

In [ ]:
class GuardrailedAgent:
    """Agent with pre/post-call guardrail enforcement."""

    def __init__(
        self,
        agent: FunctionCallingAgent,
        input_guards: Guardrails,
        output_guards: Guardrails | None = None,
    ) -> None:
        self._agent = agent
        self._input_guards = input_guards
        self._output_guards = output_guards

    async def run(self, query: str) -> str:
        # Validate input before any LLM call or tool execution
        input_result = self._input_guards.check(query)
        if not input_result.passed:
            violations = "; ".join(input_result.violations)
            return f"Request blocked: {violations}"

        answer = await self._agent.run(query)

        # Validate output before returning to the user
        if self._output_guards:
            output_result = self._output_guards.check(answer)
            if not output_result.passed:
                violations = "; ".join(output_result.violations)
                return f"Response blocked (output violation): {violations}"

        return answer

## Step 6: Wire everything together

Build the base agent and wrap it with the guardrailed wrapper.

In [ ]:
base_agent = FunctionCallingAgent(
    llm=OpenAILLM(model="gpt-4o-mini"),
    tools=[DuckDuckGoSearchTool(), CalculatorTool()],
    system_prompt="You are a helpful assistant for math and web research.",
    max_iterations=6,
)

safe_agent = GuardrailedAgent(
    agent=base_agent,
    input_guards=input_guardrails,
    output_guards=output_guardrails,
)

# Test a safe query
answer = asyncio.run(safe_agent.run("What is 12 * 34?"))
print(f"Answer: {answer}")

## Complete Working Example

A self-contained script that builds a safe agent with all three guardrails and runs it against both legitimate and blocked inputs, verifying that each is handled correctly.

In [ ]:
import asyncio
from synapsekit.agents import (
    CalculatorTool,
    ContentFilter,
    DuckDuckGoSearchTool,
    FunctionCallingAgent,
    Guardrails,
    PIIDetector,
    TopicRestrictor,
)
from synapsekit.llms.openai import OpenAILLM


class GuardrailedAgent:
    def __init__(self, agent, input_guards, output_guards=None):
        self._agent = agent
        self._input_guards = input_guards
        self._output_guards = output_guards

    async def run(self, query: str) -> str:
        result = self._input_guards.check(query)
        if not result.passed:
            return "Request blocked: " + "; ".join(result.violations)

        answer = await self._agent.run(query)

        if self._output_guards:
            out_result = self._output_guards.check(answer)
            if not out_result.passed:
                return "Response blocked: " + "; ".join(out_result.violations)

        return answer


def build_safe_agent() -> GuardrailedAgent:
    base_agent = FunctionCallingAgent(
        llm=OpenAILLM(model="gpt-4o-mini"),
        tools=[DuckDuckGoSearchTool(), CalculatorTool()],
        system_prompt="You are a helpful math and research assistant.",
        max_iterations=6,
    )

    input_guards = Guardrails(checks=[
        ContentFilter(
            blocked_patterns=[r"password\s*[:=]", r"\bAPI[_\s]?key\b"],
            blocked_words=["hack", "exploit"],
            max_length=1500,
        ),
        PIIDetector(detect=["email", "phone", "ssn"]),
        TopicRestrictor(blocked_topics=["gambling", "politics"]),
    ])

    output_guards = Guardrails(checks=[
        PIIDetector(),  # ensure the LLM didn't echo PII in its response
    ])

    return GuardrailedAgent(base_agent, input_guards, output_guards)


async def main() -> None:
    agent = build_safe_agent()

    test_inputs = [
        # Legitimate requests — should pass
        ("What is 456 * 789?", "legitimate"),
        ("What is the capital of France?", "legitimate"),
        # Blocked inputs — should be rejected
        ("My phone is 555-123-4567, help me research phone hacking", "blocked"),
        ("password: admin123 — is this secure?", "blocked"),
        ("Tell me about political gambling strategies", "blocked"),
    ]

    for query, expected in test_inputs:
        answer = await agent.run(query)
        was_blocked = answer.startswith("Request blocked") or answer.startswith("Response blocked")
        status = "OK" if (was_blocked == (expected == "blocked")) else "UNEXPECTED"
        print(f"[{status}] {query[:60]}")
        print(f"  -> {answer[:120]}")
        print()


asyncio.run(main())

## Next Steps

- [Multi-Tool Orchestration](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/multi-tool-orchestration) — add guardrails to an agent with five or more tools
- [Tool Error Handling and Retries](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/tool-error-handling) — handle tool-level errors in addition to guardrail violations
- [Streaming Agent Responses](https://synapsekit.github.io/synapsekit-docs/docs/guides/agents/streaming-agent) — show the agent's reasoning while guardrails validate the final output